In [1]:
import astropy
import pysas
import paramiko
import os
import tarfile
import subprocess
import shutil
import numpy as np
from collections import defaultdict
from astropy.io.misc import fnunpickle
from astropy.io import fits
from astropy.table import Table, vstack
from astropy.time import Time
import matplotlib.pyplot as plt
import time
import threading
#import keyboard
import pandas as pd
import scipy.optimize as opt
from matplotlib.lines import Line2D
%matplotlib qt

In [2]:
#### SPECTRA ######

working_path = "/work2/sruedavargas/Results_Sy2/NGC_5506"

In [3]:
#Starting heasoft and xmm
startsas = 'source /net/konraid/xray/setheasoft.sh'
startsasold = 'source /net/konraid/xray/setxmmsas20.0.0.sh'

subprocess.call(startsas, shell=True)
subprocess.call(startsasold, shell=True)

os.environ['SAS_DIR']='/net/konraid/xray/xmmsas/xmmsas_20.0.0'
os.environ['SAS_PATH']='/net/konraid/xray/xmmsas/xmmsas_20.0.0'
os.environ['SAS_CCFPATH']='/net/konraid/xray/xmmsas/ccf'


--- HEASOFT environment has been initialized using /net/konraid/xray/heasoft/heasoft-6.31/x86_64-pc-linux-gnu-libc2.36 .

In addition, you may need to set:  export XPA_METHOD=local .
If you prefer to get rid of the virtual python environment, type: deactivate .


--- HEASOFT environment has been initialized using /net/konraid/xray/heasoft/heasoft-6.31/x86_64-pc-linux-gnu-libc2.36 .

In addition, you may need to set:  export XPA_METHOD=local .
If you prefer to get rid of the virtual python environment, type: deactivate .


sasversion:- Executing (routine): sasversion  -w 1 -V 4
sasversion:- sasversion (sasversion-1.3)  [xmmsas_20211130_0941-20.0.0] started:  2023-09-27T16:06:40.000
sasversion:- XMM-Newton SAS release and build information:

SAS release: xmmsas_20211130_0941-20.0.0
Compiled on: Wed Dec  1 09:45:46 CET 2021
Compiled by: xmmsas@trantor
Platform   : Ubuntu20.04

SAS-related environment variables that are set:

SAS_DIR = /net/konraid/xray/xmmsas/xmmsas_20.0.0
SAS_PATH = /ne

In [4]:
products = working_path + '/products'
spectra_files = products + '/spectra_files'


os.chdir(spectra_files)

# Matching each photon event table to its corresponding gti table


phot_ev_tab = []
gti_table = []
region_file = []


for file in os.listdir(spectra_files):
    
    if file.startswith('filtered'):
        phot_ev_tab.append(file)
        
    elif 'gti_combined' in file:
        gti_table.append(file)
        
    elif file.endswith('reg.txt'):
        region_file.append(file)
        

In [5]:
# Matching the cleaned photon event table with its corresponding gti table and .txt file hat contains 
# source and background region. The .txt file with regions is read and the 4th line is taken, since it corresponds
# to the information about the source.


epat_mat=[]


for file in phot_ev_tab:
    
    obs_id_pet = f"{file.split('_')[1]}"
    cam_pet = f"{file.split('_')[2]}"
    en = os.path.splitext(file)[0]
    en_pet = f"{en.split('_')[5]}"
    row = [file]
    
    if 'MOS' in cam_pet:
        root_cam='MOS'
    if 'PN' in cam_pet:
        root_cam='PN'
        

    for file1 in gti_table:
        
        obs_id_gti = f"{file1.split('_')[0]}"
        cam_gti = f"{file1.split('_')[1]}"
        en_g = os.path.splitext(file1)[0]
        en_gti = f"{en_g.split('_')[4]}"
        
        if obs_id_pet==obs_id_gti and cam_pet==cam_gti and en_pet==en_gti:
            row.append(file1)

            for file2 in region_file:
                
                obs_id_reg = f"{file2.split('_')[0]}"
                cam_reg = f"{file2.split('_')[1]}"
                
                if obs_id_pet==obs_id_reg and root_cam==cam_reg:
                    
                    f=open(file2, "r")
                    lines=f.readlines()
                    source1=lines[3]
                    source=source1.strip()
                    
                    row.append(source)
                    
                    f.close()
                
                       
    if len(row) > 1:
        epat_mat.append(row)
        

In [6]:
# Filtering source region with the corresponding pattern in the gti
        
for element in epat_mat:
    
    table = element[0]
    gti = element[1]
    source = element[2]
    obs_id = f"{table.split('_')[1]}"
    camera = f"{table.split('_')[2]}"
    en = os.path.splitext(table)[0]
    en_pet = f"{en.split('_')[5]}"

    
    if 'MOS' in table:
        
        #MOS taken from pattern 0 to 12
        evselect_mos = [f" /net/konraid/xray/xmmsas/xmmsas_20.0.0/bin/evselect table={table} withfilteredset=true filteredset=filtered_{obs_id}_{camera}_0.2_12keV_source_reg_{en_pet}.fits keepfilteroutput=true destruct=true expression='(((X,Y) IN {source}) && GTI({gti}:STDGTI,TIME) && (PI in [200:12000]) && (PATTERN <= 12))'"]
        subprocess.call(evselect_mos, shell=True)

        
    if 'PN' in table:
        
        #PN taken only until pattern 4
        evselect_pn = [f" /net/konraid/xray/xmmsas/xmmsas_20.0.0/bin/evselect table={table} withfilteredset=true filteredset=filtered_{obs_id}_{camera}_0.2_12keV_source_reg_{en_pet}.fits keepfilteroutput=true destruct=true expression='(((X,Y) IN {source}) && GTI({gti}:STDGTI,TIME) && (PI in [200:12000]) && (PATTERN <= 4))'"]
        subprocess.call(evselect_pn, shell=True)

In [8]:
# Creating epatplots to check for pile up

%store -r prob_chip_obid

ccf_files=[]

for file in os.listdir(working_path):
    if file.endswith('.cif'):
        obs_id = f"{file.split('_')[0]}"
        ccf_files.append(file)
        if obs_id in prob_chip_obid:
            ccf_files.remove(file)
            
            
filt_source_reg=[]

for file in os.listdir(spectra_files):
    
    if 'source_reg' in file:
        filt_source_reg.append(file)
        

epatplot_files=[]

for file1 in filt_source_reg:
    obs_id_src = f"{file1.split('_')[1]}"
    row=[file1] 
    for file2 in ccf_files:
        obs_id_ccf = f"{file2.split('_')[0]}"
        if obs_id_ccf == obs_id_src:
            row.append(file2)
    if len(row)>1:
        epatplot_files.append(row)
            

for element in epatplot_files:
    file = element[0]
    ccf_file = element[1]
    os.environ['SAS_CCF'] = os.path.join(working_path, f"{ccf_file}")           

    obs_id = f"{file.split('_')[1]}"
    camera = f"{file.split('_')[2]}"
    en = os.path.splitext(file)[0]
    en_pet = f"{en.split('_')[7]}"
    
        
        
    epatplot = [f" /net/konraid/xray/xmmsas/xmmsas_20.0.0/bin/epatplot set={file} plotfile=epatplot_{obs_id}_{camera}_{en_pet}.ps"]
    subprocess.call(epatplot, shell=True) 

** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 5
** epatplot: warning (ChoiceStateVariable::set), Cannot convert 4294967295 to Node id - must be in range [0, 2]
The value -1 [UNDEFINED] for variable `node' is not among the list of allowed values: 0 [PRIMARY] 1 [REDUNDANT] 

** epatplot: warning (Mismatch), EPN/FWC_EFF is not a known instrument/filetype combination
** epatplot: warning (Mismatch), EPN/SPATIALCTI is not a known instrument/filetype combination


 %PGPLOT, Unable to read font file: grfont.dat
 %PGPLOT, Use environment variable PGPLOT_FONT to specify the location of the PGPLOT grfont.dat file.


** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 3
** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 4
** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 5
** epatplot: warning (Mismatch), EPN/FWC_EFF is not a known instrument/filetype combination
** epatplot: warning (Mismatch), EPN/SPATIALCTI is not a known instrument/filetype combination


 %PGPLOT, Unable to read font file: grfont.dat
 %PGPLOT, Use environment variable PGPLOT_FONT to specify the location of the PGPLOT grfont.dat file.


** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 5
** epatplot: warning (ChoiceStateVariable::set), Cannot convert 4294967295 to Node id - must be in range [0, 2]
The value -1 [UNDEFINED] for variable `node' is not among the list of allowed values: 0 [PRIMARY] 1 [REDUNDANT] 

** epatplot: warning (Mismatch), EPN/FWC_EFF is not a known instrument/filetype combination
** epatplot: warning (Mismatch), EPN/SPATIALCTI is not a known instrument/filetype combination


...

** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 5
** epatplot: warning (ChoiceStateVariable::set), Cannot convert 4294967295 to Node id - must be in range [0, 2]
The value -1 [UNDEFINED] for variable `node' is not among the list of allowed values: 0 [PRIMARY] 1 [REDUNDANT] 

** epatplot: warning (Mismatch), EPN/FWC_EFF is not a known instrument/filetype combination
** epatplot: warning (Mismatch), EPN/SPATIALCTI is not a known instrument/filetype combination
** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 3
** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 4
** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 5
** epatplot: warning (Mismatch), EPN/FWC_EFF is not a known instrument/filetype combination
** epatplot: warning (Mismatch), EPN/SPATIALCTI is not a known instrument/filetype combination


 %PGPLOT, Unable to read font file: grfont.dat
 %PGPLOT, Use environment variable PGPLOT_FONT to specify the location of the PGPLOT grfont.dat file.
 %PGPLOT, Unable to read font file: grfont.dat
 %PGPLOT, Use environment variable PGPLOT_FONT to specify the location of the PGPLOT grfont.dat file.


** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 5
** epatplot: warning (ChoiceStateVariable::set), Cannot convert 4294967295 to Node id - must be in range [0, 2]
The value -1 [UNDEFINED] for variable `node' is not among the list of allowed values: 0 [PRIMARY] 1 [REDUNDANT] 

** epatplot: warning (Mismatch), EPN/FWC_EFF is not a known instrument/filetype combination
** epatplot: warning (Mismatch), EPN/SPATIALCTI is not a known instrument/filetype combination
** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 5
** epatplot: warning (ChoiceStateVariable::set), Cannot convert 4294967295 to Node id - must be in range [0, 2]
The value -1 [UNDEFINED] for variable `node' is not among the list of allowed values: 0 [PRIMARY] 1 [REDUNDANT] 

** epatplot: warning (Mismatch), EPN/FWC_EFF is not a known instrument/filetype combination
** epatplot: warning (Mismatch), EPN/SPATIALCTI is not a known instrument/filetype comb

 %PGPLOT, Unable to read font file: grfont.dat
 %PGPLOT, Use environment variable PGPLOT_FONT to specify the location of the PGPLOT grfont.dat file.
 %PGPLOT, Unable to read font file: grfont.dat
 %PGPLOT, Use environment variable PGPLOT_FONT to specify the location of the PGPLOT grfont.dat file.
 %PGPLOT, Unable to read font file: grfont.dat
 %PGPLOT, Use environment variable PGPLOT_FONT to specify the location of the PGPLOT grfont.dat file.


** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 3
** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 4
** epatplot: warning (sigmaTooLarge), not enough statistics for at least 1 bin for PAT = 5
** epatplot: warning (Mismatch), EPN/FWC_EFF is not a known instrument/filetype combination
** epatplot: warning (Mismatch), EPN/SPATIALCTI is not a known instrument/filetype combination


In [9]:
# Filtering photon event tables by pattern. PN 0 and 1-4 and MOS 0-12


for file in os.listdir(spectra_files):
    
    if file.startswith('filtered') and 'source_reg' not in file:
        
        obs_id = f"{file.split('_')[1]}"
        camera = f"{file.split('_')[2]}"
        en = os.path.splitext(file)[0]
        en_pet = f"{en.split('_')[5]}"
        
        if 'MOS' in camera:
            
            evselect_mos12 = [f" /net/konraid/xray/xmmsas/xmmsas_20.0.0/bin/evselect table={file} withfilteredset=true filteredset=filtered_{obs_id}_{camera}_0.2_12keV_{en_pet}_pat12.fits keepfilteroutput=true destruct=true expression='(PATTERN <= 12) && (FLAG == 0)'"]
            subprocess.call(evselect_mos12, shell=True)
            
        if 'PN' in camera:
            
            evselect_pn0 = [f" /net/konraid/xray/xmmsas/xmmsas_20.0.0/bin/evselect table={file} withfilteredset=true filteredset=filtered_{obs_id}_{camera}_0.2_12keV_{en_pet}_pat0.fits keepfilteroutput=true destruct=true expression='(PATTERN == 0) && (FLAG == 0)'"]
            subprocess.call(evselect_pn0, shell=True)


            evselect_pn14 = [f" /net/konraid/xray/xmmsas/xmmsas_20.0.0/bin/evselect table={file} withfilteredset=true filteredset=filtered_{obs_id}_{camera}_0.2_12keV_{en_pet}_pat14.fits keepfilteroutput=true destruct=true expression='((PATTERN > 0) && (PATTERN <= 4)) && (FLAG == 0)'"]
            subprocess.call(evselect_pn14, shell=True)
        

In [10]:
# Extracting spectra. Defining source and background region expresion

pet_pat_spec=[]
bin_gti_spec=[]
comb_gti_spec=[]
reg_file=[]



for file in os.listdir(spectra_files):
    if file.startswith('filtered') and ('source_reg' not in file) and ('pat' in file):
        pet_pat_spec.append(file)
        
    elif file.startswith('bin_gti') and file.endswith('.fits'):
        bin_gti_spec.append(file)
        
    elif file.endswith('reg.txt'):
        reg_file.append(file)
    
    elif ('gti_combined' in file) and (file.endswith('.fits')):
        comb_gti_spec.append(file)



espec_mat=[]     
        
for file1 in pet_pat_spec:  
    obs_id_pet = f"{file1.split('_')[1]}"
    camera_pet = f"{file1.split('_')[2]}"
    en1 = os.path.splitext(file1)[0]
    en_pet = f"{en1.split('_')[5]}"
    row=[file1]
    
    if 'MOS' in camera_pet:
        root_cam='MOS'
    if 'PN' in camera_pet:
        root_cam='PN'
 
    for file2 in bin_gti_spec:
        obs_id_gti = f"{file2.split('_')[2]}"
        camera_gti = f"{file2.split('_')[3]}"
        en2 = os.path.splitext(file2)[0]
        en_gti = f"{en2.split('_')[4]}"
        
        if obs_id_pet==obs_id_gti and en_pet==en_gti:
            row.append(file2)
            
            for file3 in ccf_files:
                obs_id_ccf = f"{file3.split('_')[0]}"
                if obs_id_ccf == obs_id_pet:
                    row.append(file3)
                    
                    for file4 in comb_gti_spec:              
                        obs_id_comb_gti = f"{file4.split('_')[0]}"
                        camera_comb_gti = f"{file4.split('_')[1]}"
                        en_comb = os.path.splitext(file4)[0]
                        en_comb_gti = f"{en_comb.split('_')[4]}"
                        
                        if obs_id_pet==obs_id_comb_gti and camera_pet==camera_comb_gti and en_pet==en_comb_gti:
                            row.append(file4)
                        
           
                            for file5 in reg_file:

                                obs_id_reg = f"{file5.split('_')[0]}"
                                camera_reg = f"{file5.split('_')[1]}"

                                if obs_id_pet==obs_id_reg and root_cam==camera_reg:

                                    f=open(file5, "r")
                                    lines=f.readlines()

                                    source1=lines[3]
                                    source=source1.strip()

                                    source_exp=f"((X,Y) IN {source})"

                                    exp = ""
                                    for line in lines:
                                        if '# background' in line:
                                            line_c = f"{line.split('#')[0]}"
                                            back_temp = f"((X,Y) IN {line_c}) ||"
                                            exp += back_temp

                                    background_exp=exp[:-2]     
                                    f.close()

                                    row.append(source_exp)
                                    row.append(background_exp)
                    
    if len(row)>1:
        
        espec_mat.append(row)

                

In [11]:
# Extracting averaged spectra

for element in espec_mat:
    
    ccf_file = element[2]
    os.environ['SAS_CCF'] = os.path.join(working_path, f"{ccf_file}") 
    
    table = element[0]
    binned_gti = element[1]
    combined_gti = element[3]
    source_exp = element[4]
    background_exp = element[5]
    
    source_exp_comb_gti = f"({source_exp} && GTI({combined_gti}:STDGTI,TIME))"
    background_exp_comb_gti = f"({background_exp} && GTI({combined_gti}:STDGTI,TIME))"
    
    obs_id = f"{table.split('_')[1]}"
    camera = f"{table.split('_')[2]}"
    en = f"{table.split('_')[5]}"
    pat = os.path.splitext(table)[0]
    pattern = f"{pat.split('_')[6]}"
    
    #Averaged spectra
    especget_av = [f" /net/konraid/xray/xmmsas/xmmsas_20.0.0/bin/especget filestem=av_spec_{obs_id}_{camera}_{en}_{pattern} srcexp='{source_exp_comb_gti}' backexp='{background_exp_comb_gti}' table={table}"]
    subprocess.call(especget_av, shell=True)

** especget::evselect: warning (ExpBadDssGlobal), A filtering expression has been specified which may not allow for calculation of a uniform exposure for all events.  Please ensure that all TIME related filters are applied uniformly.


Set to: notSpecified


** especget::rmfgen: warning (Mismatch), EPN/FWC_EFF is not a known instrument/filetype combination
** especget::rmfgen: warning (Mismatch), EPN/SPATIALCTI is not a known instrument/filetype combination


Making file temp_badcol.ds
Making file temp_badcol.ds


** especget::arfgen: warning (zeroSumDetmap), The filtered detector map has a zero normalisation - returning zero area. Try using a flat detector map and/or more bins
** especget::evselect: warning (DssBlockClash), Name clash while writing Data Subspace: Block GTI00006 already exists in output data set. Renaming block to GTI00007.
** especget::evselect: warning (DssBlockClash), Name clash while writing Data Subspace: Block GTI00106 already exists in output data set. Renaming block to GTI00107.
** especget::evselect: warning (DssBlockClash), Name clash while writing Data Subspace: Block GTI00206 already exists in output data set. Renaming block to GTI00207.
** especget::evselect: warning (DssBlockClash), Name clash while writing Data Subspace: Block GTI00306 already exists in output data set. Renaming block to GTI00307.
** especget::evselect: warning (DssBlockClash), Name clash while writing Data Subspace: Block GTI00406 already exists in output data set. Renaming block to GTI00407.
** 

...

** especget::arfgen: warning (zeroSumDetmap), The filtered detector map has a zero normalisation - returning zero area. Try using a flat detector map and/or more bins
** especget::evselect: warning (ExpBadDssGlobal), A filtering expression has been specified which may not allow for calculation of a uniform exposure for all events.  Please ensure that all TIME related filters are applied uniformly.


Set to: notSpecified


** especget::rmfgen: warning (Mismatch), EPN/FWC_EFF is not a known instrument/filetype combination
** especget::rmfgen: warning (Mismatch), EPN/SPATIALCTI is not a known instrument/filetype combination


Making file temp_badcol.ds
Making file temp_badcol.ds


** especget::arfgen: warning (zeroSumDetmap), The filtered detector map has a zero normalisation - returning zero area. Try using a flat detector map and/or more bins


In [12]:
# Extracting single spectra

for element in espec_mat:
    
    ccf_file = element[2]
    os.environ['SAS_CCF'] = os.path.join(working_path, f"{ccf_file}") 
    
    table = element[0]
    binned_gti = element[1]
    combined_gti = element[3]
    source_exp = element[4]
    background_exp = element[5]
    
    obs_id = f"{table.split('_')[1]}"
    camera = f"{table.split('_')[2]}"
    en = f"{table.split('_')[5]}"
    pat = os.path.splitext(table)[0]
    pattern = f"{pat.split('_')[6]}"
    
    
    table_bin_gti = fits.open(binned_gti)
    data = table_bin_gti[1].data
    
    n=0
    
    for inter in data:
        
        n += 1
        start=inter[0]
        stop=inter[1]
 
        source_exp_bin_gti = f"({source_exp} && (TIME >= {start}) && (TIME <= {stop}))"
        background_exp_bin_gti = f"({background_exp} && (TIME >= {start}) && (TIME <= {stop}))"


        #Single spectra

        especget_sing = [f" /net/konraid/xray/xmmsas/xmmsas_20.0.0/bin/especget filestem=sing_spec_{obs_id}_{camera}_{en}_{pattern}_{n} srcexp='{source_exp_bin_gti}' backexp='{background_exp_bin_gti}' table={table}"]
        subprocess.call(especget_sing, shell=True)
    

        

** especget::evselect: warning (ExpBadDssGlobal), A filtering expression has been specified which may not allow for calculation of a uniform exposure for all events.  Please ensure that all TIME related filters are applied uniformly.


Set to: notSpecified


** especget::rmfgen: warning (Mismatch), EPN/FWC_EFF is not a known instrument/filetype combination
** especget::rmfgen: warning (Mismatch), EPN/SPATIALCTI is not a known instrument/filetype combination


Making file temp_badcol.ds
Making file temp_badcol.ds


** especget::arfgen: warning (zeroSumDetmap), The filtered detector map has a zero normalisation - returning zero area. Try using a flat detector map and/or more bins
** especget::evselect: warning (ExpBadDssGlobal), A filtering expression has been specified which may not allow for calculation of a uniform exposure for all events.  Please ensure that all TIME related filters are applied uniformly.


...

** especget::arfgen: warning (zeroSumDetmap), The filtered detector map has a zero normalisation - returning zero area. Try using a flat detector map and/or more bins
** especget::evselect: warning (ExpBadDssGlobal), A filtering expression has been specified which may not allow for calculation of a uniform exposure for all events.  Please ensure that all TIME related filters are applied uniformly.


Set to: notSpecified


** especget::rmfgen: warning (Mismatch), EPN/FWC_EFF is not a known instrument/filetype combination
** especget::rmfgen: warning (Mismatch), EPN/SPATIALCTI is not a known instrument/filetype combination


Making file temp_badcol.ds
Making file temp_badcol.ds


** especget::arfgen: warning (zeroSumDetmap), The filtered detector map has a zero normalisation - returning zero area. Try using a flat detector map and/or more bins


In [13]:
#  Grouping the spectra


for file in os.listdir(spectra_files):
    
    if file.endswith('_src.ds'):
        
        file_name = os.path.splitext(file)[0]
        res = f"{file_name}_grp20.ds"
        
        
        if 'PN' in file:
            
            specgroup_pn = [f"/net/konraid/xray/xmmsas/xmmsas_20.0.0/bin/specgroup spectrumset={file} groupedset={res} mincounts=20 setbad='0:38,2400:4096'"]
            subprocess.call(specgroup_pn, shell=True)

        if 'MOS' in file:
            
            specgroup_mos = [f"/net/konraid/xray/xmmsas/xmmsas_20.0.0/bin/specgroup spectrumset={file} groupedset={res} mincounts=20 setbad='0:14,799:800'"]
            subprocess.call(specgroup_mos, shell=True)
            

** specgroup: warning (AlreadyGrouped), Bin 4096 has already been grouped and will not be set to bad
** specgroup: warning (AlreadyGrouped), Bin 4096 has already been grouped and will not be set to bad
** specgroup: warning (NoMergeGroup), Cant find group to merge with - creating a new group (39:2399)
** specgroup: warning (AlreadyGrouped), Bin 4096 has already been grouped and will not be set to bad
** specgroup: warning (AlreadyGrouped), Bin 4096 has already been grouped and will not be set to bad
** specgroup: warning (AlreadyGrouped), Bin 4096 has already been grouped and will not be set to bad
** specgroup: warning (AlreadyGrouped), Bin 4096 has already been grouped and will not be set to bad
** specgroup: warning (AlreadyGrouped), Bin 4096 has already been grouped and will not be set to bad
** specgroup: warning (AlreadyGrouped), Bin 4096 has already been grouped and will not be set to bad
** specgroup: warning (AlreadyGrouped), Bin 4096 has already been grouped and will not be s